# Домашняя работа №6 — кластеризация

**Данные:** ~2240 клиентов розничной сети — возраст, доход, траты по категориям, каналы покупок.

**Задача:** разбить клиентов на группы, чтобы каждую можно было назвать одним словом — «экономные», «премиум», «семейные».

Код в модулях (`modules/`). `main.py` прогоняет всё и сохраняет графики в `output/`. Здесь — по шагам вручную с отображением графиков в ноутбуке.


## 1. Импорт и загрузка

Если `data/marketing_campaign.csv` нет — скачается сам.

In [ ]:
%matplotlib inline

from pathlib import Path

from modules import (
    ClusterBench,
    ClusterEvaluator,
    ClusterVisualizer,
    CustomerDataset,
    CustomerPreprocessor,
    HypothesisTester,
)
from modules.preprocessing import SPENDING_COLS

ROOT = Path('.').resolve()
DATA = ROOT / 'data'
OUT = ROOT / 'output'
K_PREFERRED = 4
RANDOM_STATE = 42

dataset = CustomerDataset(DATA)
preprocessor = CustomerPreprocessor()
evaluator = ClusterEvaluator(random_state=RANDOM_STATE)
bench = ClusterBench(random_state=RANDOM_STATE)
viz = ClusterVisualizer(OUT)
hyp_tester = HypothesisTester()

raw = dataset.load()
raw.head()


## 2. EDA — распределения, выбросы, корреляции

In [ ]:
eda = preprocessor.describe_eda(raw)
print('Пропуски:', dict(eda['missing']))
print('Константные:', eda['const_cols'])
print('\nТоп по range:')
eda['scale'].head(8)


In [ ]:
viz.plot_numeric_distributions(raw, list(eda['num_cols']))
viz.plot_spending_boxplots(raw, SPENDING_COLS)
viz.plot_correlation_heatmap(raw, list(eda['num_cols']))


## 3–6. Предобработка

- **Выбросы:** IQR-winsorization (не удаляем строки — K-Means чувствителен)
- **Мультиколлинеарность:** убираем признаки с |r|>0.8 к агрегатам
- **Family_Size:** Kidhome + Teenhome + Adults (2 при Married/Together, иначе 1)
- **Масштабирование:** RobustScaler (не усиливает выбросы как StandardScaler)


In [ ]:
prepared = preprocessor.prepare(raw)
print(f'Признаков: {len(prepared.feature_names)}')
print(f'Скейлер: {prepared.scaler_name}')
print(f'Удалено (мультиколлинеарность): {list(prepared.dropped_cols)}')
print(f'Обрезано выбросов (IQR): {sum(prepared.outlier_stats.values())}')
prepared.processed[['Age', 'Income', 'Family_Size', 'Adults_In_Household', 'Total_Spending']].head()


## Влияние выбросов на K-Means

Сравнение silhouette: IQR-cap + RobustScaler vs без обработки + StandardScaler.

In [ ]:
prepared_raw = preprocessor.prepare(
    raw, handle_outliers=False, drop_multicollinear=True, scaler='standard'
)
k_search_robust = evaluator.search_k(prepared.x, k_min=2, k_max=10)
k_search_raw = evaluator.search_k(prepared_raw.x, k_min=2, k_max=10)
viz.plot_outlier_impact(
    k_search_robust.k_range,
    k_search_robust.silhouettes,
    k_search_raw.silhouettes,
)


## Гипотезы и проверки (H1–H6)

| ID | Гипотеза |
|----|----------|
| H1 | Чем выше доход, тем реже покупают по скидкам |
| H2 | Доход влияет на предпочтение веб-канала |
| H3 | Семьи с детьми тратят больше на мясо |
| H4 | Возраст связан с давностью покупки (Recency) |
| H5 | Образование влияет на траты на вино |
| H6 | Верхний квартиль дохода реже использует скидки |


In [ ]:
hyp_results = hyp_tester.run_all(prepared.processed)
hyp_table = hyp_tester.summary_table(hyp_results)
hyp_table[['id', 'hypothesis', 'test', 'p_value', 'significant', 'conclusion']]


In [ ]:
viz.plot_hypothesis_income_deals(prepared.processed)
viz.plot_hypothesis_channels(prepared.processed)
viz.plot_hypothesis_family_meat(prepared.processed)


## 8. Выбор числа кластеров (Elbow + Silhouette)

`pick_k`: preferred k, если silhouette ≥ 90% от максимума; иначе — best_sil.

In [ ]:
k_pick = evaluator.pick_k(k_search_robust, preferred=K_PREFERRED, silhouette_threshold=0.90)
k_best = k_pick.k
print(k_pick.reason)
viz.plot_k_search(
    k_search_robust.k_range,
    k_search_robust.inertias,
    k_search_robust.silhouettes,
    k_best=k_best,
)


## 7. Кластеризация: K-Means → Spectral

In [ ]:
results = bench.run_all(prepared.x, k=k_best)
summary = bench.summary_table()
summary


In [ ]:
gmm = results['GMM'].extra
print(f"GMM: признаков={gmm['n_features']}, параметров≈{gmm['n_params']}")
print(f"params/sample={gmm['params_per_sample']}, BIC={gmm['bic']:.0f}, AIC={gmm['aic']:.0f}")
print(f"DBSCAN eps={results['DBSCAN'].extra.get('eps', '—'):.3f} (авто, k-distance graph)")

viz.plot_k_distance(prepared.x, min_samples=10, eps=results['DBSCAN'].extra.get('eps'))
viz.plot_dendrogram(prepared.x, random_state=RANDOM_STATE)
viz.plot_methods_comparison(summary)


In [ ]:
labels = results['K-Means'].labels
profile = evaluator.cluster_profile(prepared.processed, labels, prepared.profile_cols)
profile


## 9. Аномалии

In [ ]:
anomalies = evaluator.detect_anomalies(
    prepared.x, labels,
    results['DBSCAN'].labels,
    results['HDBSCAN'].labels,
)
anomalies.counts


In [ ]:
viz.plot_anomalies_pca(
    prepared.x,
    {
        'DBSCAN шум': results['DBSCAN'].labels == -1,
        'Isolation Forest': anomalies.overlap['isolation_forest'].to_numpy(),
        'LOF': anomalies.overlap['lof'].to_numpy(),
    },
    random_state=RANDOM_STATE,
)


## 10. Визуализация (2D + 3D)

PCA / t-SNE / UMAP / 3D — только для картинок, кластеризация в полном пространстве.

In [ ]:
viz.plot_projections(prepared.x, labels, k=k_best, random_state=RANDOM_STATE)


In [ ]:
viz.plot_3d_pca(prepared.x, labels, k=k_best, random_state=RANDOM_STATE)
viz.plot_3d_features(prepared.processed, labels)


In [ ]:
viz.plot_silhouette(prepared.x, labels, k=k_best)
viz.plot_radar_profiles(profile, list(prepared.profile_cols))
viz.plot_parallel_coordinates(prepared.processed, labels, list(prepared.profile_cols))


## Итог

- **Выбранное k:** см. вывод `pick_k` выше
- **Гипотез подтверждено:** см. таблицу H1–H6
- **Профили кластеров:** Income, Family_Size, Total_Spending в таблице `profile`
- Все графики также сохраняются в `output/`
